In [ ]:
%pip install python-dotenv requests gradio pandas

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()   

DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
WORKSPACE_URL    = os.getenv("WORKSPACE_URL")
SERVING_ENDPOINT = os.getenv("SERVING_ENDPOINT")

ENDPOINT_URL = f"https://{WORKSPACE_URL}/serving-endpoints/{SERVING_ENDPOINT}/invocations"
HEADERS      = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}


print(f"Workspace : {WORKSPACE_URL}")
print(f"Endpoint  : {SERVING_ENDPOINT}")
print(f"Endpoint URL: {ENDPOINT_URL}")
print(f"Token set : {'Yes' if DATABRICKS_TOKEN else 'NO - check .env file'}")

# Core function

In [ ]:
import requests
import json

def chat_with_rag(question: str, region: str = None) -> dict:
    payload = {
        "dataframe_records": [
            {
                "question": question,
                "region"  : region if region else None
            }
        ]
    }

    response = requests.post(
        ENDPOINT_URL,
        headers = HEADERS,
        json    = payload,
        timeout = 180
    )

    if response.status_code != 200:
        return {"error": f"HTTP {response.status_code}: {response.text}"}

    prediction = response.json()["predictions"][0]

    # Parse citations — comes back as JSON string
    raw_citations = prediction.get("citations", "[]")
    citations     = json.loads(raw_citations) if isinstance(raw_citations, str) else raw_citations

    raw_mappable  = prediction.get("mappable_facilities", "[]")
    mappable      = json.loads(raw_mappable) if isinstance(raw_mappable, str) else raw_mappable

    return {
        "answer"             : prediction.get("answer", ""),
        "citations"          : citations,
        "show_map_button"    : prediction.get("show_map_button", False),
        "mappable_facilities": mappable,
        "num_sources"        : prediction.get("num_sources", 0)
    }

# Test to verify connection

In [ ]:
# ============================================================
# Quick test — run this first to verify connection
# ============================================================
result = chat_with_rag("What services does 2BN Military Hospital offer?")

print(f"Answer:\n{result['answer']}")
print(f"\nSources used: {result['num_sources']}")
print(f"Show map    : {result['show_map_button']}")
print(f"\nCitations:")
for c in result["citations"]:
    print(f"  [{c['rank']}] {c['name']} · {c['city']}, {c['region']} · score: {c['relevance_score']}")

Answer:
2BN Military Hospital, located in Apremdo, Western, Ghana, offers services in internalMedicine, dentistry, and emergencyMedicine.

Sources used: 3
Show map    : True

Citations:
  [1] 2BN Military Hospital · Apremdo, Western · score: 0.6764
  [2] 37 Military Hospital · Accra,  · score: 0.6295
  [3] War Memorial Hospital · Nogsenia,  · score: 0.6246


# Gradio UI

In [ ]:
import gradio as gr

REGIONS = [
    "",
    "Greater Accra Region",
    "Ashanti Region",
    "Western Region",
    "Northern Region",
    "Eastern Region",
    "Central Region",
    "Upper East Region",
    "Upper West Region",
    "Volta Region",
    "Bono Region",
]

def format_response(result: dict) -> str:
    if "error" in result:
        return f"Error: {result['error']}"

    answer    = result["answer"]
    citations = result["citations"]
    show_map  = result["show_map_button"]

    formatted  = f"{answer}\n\n"

    if citations:
        formatted += "**Source Facilities:**\n"
        for c in citations:
            loc_flag  = "" if c.get("has_location") else " *(no GPS)*"
            formatted += (
                f"\n**[{c['rank']}] {c.get('name')}**{loc_flag}\n"
                f"- Location : {c.get('city')}, {c.get('region')}\n"
                f"- Type     : {c.get('facility_type')}\n"
                f"- Score    : {c.get('relevance_score', 0):.4f}\n"
            )

    if show_map:
        formatted += "\n*Map view available for this query*\n"

    formatted += f"\n*Sources used: {result['num_sources']}*"
    return formatted

def respond(message, history, region):
    result = chat_with_rag(message, region if region else None)
    return format_response(result)

with gr.Blocks(title="Ghana Medical Facility Assistant") as demo:
    gr.Markdown("## Ghana Medical Facility Assistant")
    gr.Markdown("Ask about healthcare facilities across Ghana. Powered by Virtue Foundation data.")

    with gr.Row():
        region_filter = gr.Dropdown(
            choices = REGIONS,
            value   = "",
            label   = "Filter by region (optional)",
            scale   = 1
        )

    gr.ChatInterface(
        fn                = respond,
        additional_inputs = [region_filter],
        examples          = [
            ["What services does 2BN Military Hospital offer?", ""],
            ["Which hospitals have emergency care?",             "Greater Accra Region"],
            ["Find facilities with maternity services",          "Ashanti Region"],
            ["Are there any hospitals with ICU beds?",           ""],
        ],
        title = "",
    )

demo.launch()   